In [20]:
import pandas as pd

# Load the fail probabilities produced by the model in 02_modeling.ipynb
df_opt = pd.read_csv('inspection_candidates.csv')
df_opt.shape

(314, 38)

In [21]:
import gurobipy as gp
from gurobipy import GRB

## Problem Formulation

**Goal**: Given limited inspection capacity, decide which units to 
inspect in order to minimize the expected cost of missed fails.

**Decision variable**: 
- x_i = 1 if unit i is inspected, 0 otherwise

**Objective**: Minimize expected cost from uninspected units
- minimize Σ (1 - x_i) × p_i × C_miss

**Constraint**: Inspection capacity limit
- Σ x_i ≤ K

Where:
- p_i = predicted fail probability for unit i
- C_miss = cost of missing one fail unit (assumed)
- K = maximum number of units that can be inspected

In [36]:
# Create a new optimization model
model = gp.Model("inspection_allocation")

n_units = len(df_opt)
fail_prob = df_opt["fail_probability"].values

In [37]:
# Decision variable: x[i] = 1 if unit i is inspected, 0 otherwise
# vtype=GRB.BINARY restricts each x[i] to only 0 or 1 
x = model.addVars(n_units, vtype=GRB.BINARY, name="inspect")

# Capacity constraint: total number of inspected units cannot exceed K
K = 50
model.addConstr(gp.quicksum(x[i] for i in range (n_units)) <= K, name = "capacity")

# Cost parameter: cost of missing one fail unit
C_miss = 100

In [38]:
# Objective: minimize expected cost from units NOT inspected
# (1 - x[i]) is 1 when NOT inspected, 0 when inspected

model.setObjective(
    gp.quicksum((1 - x[i]) * fail_prob[i] * C_miss for i in range (n_units)),
    GRB.MINIMIZE
)
model.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-1360P, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 1 rows, 314 columns and 314 nonzeros (Min)
Model fingerprint: 0x537c9dd2
Model has 314 linear objective coefficients and an objective constant of 5420
Variable types: 0 continuous, 314 integer (314 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 5e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e+01, 5e+01]

Found heuristic solution: objective 4711.0000000
Presolve removed 1 rows and 314 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.03 seconds (0.00 work units)
Thread count was 1 (of 16 available processors)

Solution count 2: 3801 4711 

Optimal solution found (tolerance 1.00e-04)
Best objective 3.801

## Solve the Optimization Problem

Just ran the Gurobi solver to find the optimal set of units to inspect, 
then extract which units were selected.

In [23]:
# Extract which units were selected for inspection (x[i] = 1)
selected = [i for i in range(n_units) if x[i].X > 0.5]
print(f"Number of units selected: {len(selected)}")
print(f"Fail probabilities selected units: \n{df_opt.iloc[selected]['fail_probability'].describe()}")

Number of units selected: 50
Fail probabilities selected units: 
count    50.000000
mean      0.323800
std       0.057743
min       0.260000
25%       0.280000
50%       0.305000
75%       0.347500
max       0.520000
Name: fail_probability, dtype: float64


## Revisiting the Cost Assumption
- First attempt: simple count constraint (K units) → same result as 
  sorting by fail probability. Gurobi added no value over a basic sort.
- Fix: introduce cost heterogeneity to make this a real 0/1 knapsack 
  problem (where greedy sort ≠ optimal).
- Rejected idea: base cost on semiconductor inspection types 
  — no domain expertise to defend this.
- Adopted approach: random inspection cost within a realistic range, 
  independent of fail probability. Explicitly a simplifying assumption 
  (no real cost structure claimed) made to demonstrate the knapsack 
  formulation itself.

## Cost Assumption (Simplified)

Rather than deriving inspection cost from a specific dataset feature 
(which turned out to be unreliable in this test subset), inspection 
cost is modeled as a random value within a realistic range, independent 
of fail probability. This is explicitly stated as a simplifying 
assumption for demonstrating the optimization method, not a claim 
about real inspection cost structures.

In [28]:
import numpy as np
np.random.seed(42)

# Assign a random inspection cost per unit, independent of fail probability,
# as a simplifying assumption to create a genuine knapsack problem
df_opt['inspection_cost'] = np.random.uniform(5, 20, len(df_opt))
df_opt[['fail_probability', 'inspection_cost']].describe()

,fail_probability,inspection_cost
count,314.000000,314.000000
mean,0.172611,12.425245
std,0.086985,4.421476
min,0.010000,5.075924
25%,0.110000,8.602017
50%,0.160000,12.697456
75%,0.220000,16.299293
max,0.520000,19.850808


## Solve as a 0/1 Knapsack Problem

- Decision: x_i = 1 if unit i is inspected
- Objective: minimize expected cost of missed fails (same as before)
- Constraint: total inspection COST (not count) must stay within budget

In [30]:
model2 = gp.Model("inspection_knapsack")

n_units = len(df_opt)
fail_prob = df_opt['fail_probability'].values
cost = df_opt['inspection_cost'].values

# Decision variable: same as before, 1 = inspect, 0 = skip
x = model2.addVars(n_units, vtype=GRB.BINARY, name="inspect")

# Budget constraint: total cost of inspected units must stay within budget
# (this replaces the simple "K units" count constraint from before)
budget = 1000
model2.addConstr(gp.quicksum(cost[i] * x[i] for i in range(n_units)) <= budget, name="budget")

# Objective: same as before, minimize expected cost from uninspected units
C_miss = 100
model2.setObjective(
    gp.quicksum((1 - x[i]) * fail_prob[i] * C_miss for i in range(n_units)),
    GRB.MINIMIZE
)

model2.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-1360P, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 1 rows, 314 columns and 314 nonzeros (Min)
Model fingerprint: 0xd3393210
Model has 314 linear objective coefficients and an objective constant of 5420
Variable types: 0 continuous, 314 integer (314 binary)
Coefficient statistics:
  Matrix range     [5e+00, 2e+01]
  Objective range  [1e+00, 5e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+03, 1e+03]

Found heuristic solution: objective 4121.0000000
Presolve time: 0.00s
Presolved: 1 rows, 314 columns, 314 nonzeros
Variable types: 0 continuous, 314 integer (314 binary)

Root relaxation: objective 2.796240e+03, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | 

## Verify: Does Gurobi Actually Outperform a Simple Greedy Sort?

Compare Gurobi's optimal solution against the naive strategy of 
sorting units by fail probability and greedily selecting until the 
budget runs out.

In [31]:
# Greedy baseline: sort by fail probability descending, 
# keep adding units until the budget is used up
df_sorted = df_opt.sort_values('fail_probability', ascending=False).reset_index(drop=True)

greedy_selected = []
running_cost = 0
for i in range(len(df_sorted)):
    if running_cost + df_sorted.loc[i, 'inspection_cost'] <= budget:
        greedy_selected.append(df_sorted.loc[i, 'fail_probability'])
        running_cost += df_sorted.loc[i, 'inspection_cost']

# Compute the greedy strategy's expected cost using the same objective formula
greedy_missed_prob_sum = df_sorted['fail_probability'].sum() - sum(greedy_selected)
greedy_objective = greedy_missed_prob_sum * C_miss

print(f"Greedy objective: {greedy_objective:.0f}")
print(f"Gurobi objective: {model2.ObjVal:.0f}")
print(f"Improvement: {greedy_objective - model2.ObjVal:.0f}")

Greedy objective: 3075
Gurobi objective: 2797
Improvement: 278


## Result: Gurobi vs. Greedy Baseline

- Greedy (sort by fail probability, fill budget): objective = 3075
- Gurobi (true optimal combination): objective = 2797
- Improvement: ~9% reduction in expected cost, same budget

This confirms the knapsack formulation was not just theoretically 
correct but practically meaningful: greedy sorting leaves value on 
the table that Gurobi's global search recovers.

In [32]:
# Extract which units Gurobi selected for inspection
gurobi_selected = [1 if x[i].X > 0.5 else 0 for i in range(n_units)]

df_opt['gurobi_inspected'] = gurobi_selected

In [33]:
# Reconstruct greedy selection as a column too, for the same comparison
df_sorted['greedy_inspected'] = 0
running_cost = 0
for i in range(len(df_sorted)):
    if running_cost + df_sorted.loc[i, 'inspection_cost'] <= budget:
        df_sorted.loc[i, 'greedy_inspected'] = 1
        running_cost += df_sorted.loc[i, 'inspection_cost']

In [34]:
# Merge greedy result back into df_opt (they share the same units, 
# just in a different row order)
df_opt = df_opt.merge(
    df_sorted[['fail_probability', 'inspection_cost', 'greedy_inspected']],
    on=['fail_probability', 'inspection_cost'],
    how='left'
)

In [35]:
df_opt.to_csv('../tableau_export.csv', index=False)
print("Saved tableau_export.csv")

Saved tableau_export.csv


In [39]:
# Count how many actual fails were missed by each strategy
greedy_missed = ((df_opt['actual_label'] == 1) & (df_opt['greedy_inspected'] == 0)).sum()
gurobi_missed = ((df_opt['actual_label'] == 1) & (df_opt['gurobi_inspected'] == 0)).sum()

print(f"Greedy missed: {greedy_missed} fails")
print(f"Gurobi missed: {gurobi_missed} fails")

Greedy missed: 9 fails
Gurobi missed: 7 fails


In [40]:
# Save this comparison as a simple summary table for the main Tableau chart
summary = pd.DataFrame({
    'strategy': ['Greedy', 'Gurobi'],
    'fails_caught': [21 - greedy_missed, 21 - gurobi_missed],
    'fails_missed': [greedy_missed, gurobi_missed]
})
summary.to_csv('../tableau_summary.csv', index=False)
summary

,strategy,fails_caught,fails_missed
0,Greedy,12,9
1,Gurobi,14,7
